In [27]:
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Load Data
x_data = pd.read_csv("x_data.csv")
y_data = pd.read_csv("y_data.csv")

# Standardize column names to avoid errors caused by leading or trailing spaces
x_data.columns = x_data.columns.str.strip()
y_data.columns = y_data.columns.str.strip()

print("x_data shape:", x_data.shape)
print("y_data shape:", y_data.shape)

print("\n[x_data columns]")
print(x_data.columns.tolist())

print("\n[y_data columns]")
print(y_data.columns.tolist())

display(x_data.head())
display(y_data.head())

x_data shape: (817, 7)
y_data shape: (2960704, 9)

[x_data columns]
['Date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']

[y_data columns]
['PERMNO', 'HdrCUSIP', 'Ticker', 'PERMCO', 'MthCalDt', 'MthPrc', 'MthRet', 'MthVol', 'ShrOut']


,Date,Mkt-RF,SMB,HML,RMW,CMA,RF
0,196307,-0.39,-0.48,-0.81,0.64,-1.15,0.27
1,196308,5.08,-0.80,1.70,0.40,-0.38,0.25
2,196309,-1.57,-0.43,0.00,-0.78,0.15,0.27
3,196310,2.54,-1.34,-0.04,2.79,-2.25,0.29
4,196311,-0.86,-0.85,1.73,-0.43,2.27,0.27


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthRet,MthVol,ShrOut
0,10001,36720410,EWST,7953,1995-01-31,7.750000,-0.031250,27232.0,2224
1,10001,36720410,EWST,7953,1995-02-28,7.546875,-0.026210,21290.0,2224
2,10001,36720410,EWST,7953,1995-03-31,7.500000,0.005971,8246.0,2244
3,10001,36720410,EWST,7953,1995-04-28,7.500000,0.000000,18812.0,2244
4,10001,36720410,EWST,7953,1995-05-31,7.875000,0.050000,19932.0,2244


In [28]:
# =========================
# 1. process data: DateKey
# y_data: MthCalDt -> datetime -> YYYYMM
# x_data: Date is already YYYYMM
# =========================
y_data["MthCalDt"] = pd.to_datetime(y_data["MthCalDt"], errors="coerce")
y_data["DateKey"] = y_data["MthCalDt"].dt.year * 100 + y_data["MthCalDt"].dt.month

x_data["Date"] = pd.to_numeric(x_data["Date"], errors="coerce").astype("Int64")
x_data["DateKey"] = x_data["Date"]

# =========================
# Scale the FF5 factors and the risk-free rate from percentages to decimals
# =========================
factor_cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
for col in factor_cols:
    x_data[col] = pd.to_numeric(x_data[col], errors="coerce") / 100.0

# =========================
# Clean the price variable: CRSP prices may be recorded as negative, so take the absolute value
# =========================
y_data["MthPrc"] = pd.to_numeric(y_data["MthPrc"], errors="coerce")
y_data["MthPrc_Abs"] = y_data["MthPrc"].abs()

# Convert other key columns to numeric format
numeric_cols_y = ["PERMNO", "PERMCO", "MthRet", "MthVol", "ShrOut"]
for col in numeric_cols_y:
    if col in y_data.columns:
        y_data[col] = pd.to_numeric(y_data[col], errors="coerce")

print("Date and scaling finished.")
display(y_data[["MthCalDt", "DateKey", "MthPrc", "MthPrc_Abs"]].head())
display(x_data[["Date", "DateKey"] + factor_cols].head())

Date and scaling finished.


,MthCalDt,DateKey,MthPrc,MthPrc_Abs
0,1995-01-31,199501,7.750000,7.750000
1,1995-02-28,199502,7.546875,7.546875
2,1995-03-31,199503,7.500000,7.500000
3,1995-04-28,199504,7.500000,7.500000
4,1995-05-31,199505,7.875000,7.875000


,Date,DateKey,Mkt-RF,SMB,HML,RMW,CMA,RF
0,196307,196307,-0.0039,-0.0048,-0.0081,0.0064,-0.0115,0.0027
1,196308,196308,0.0508,-0.0080,0.0170,0.0040,-0.0038,0.0025
2,196309,196309,-0.0157,-0.0043,0.0000,-0.0078,0.0015,0.0027
3,196310,196310,0.0254,-0.0134,-0.0004,0.0279,-0.0225,0.0029
4,196311,196311,-0.0086,-0.0085,0.0173,-0.0043,0.0227,0.0027


In [29]:
# =========================
# Merge the stock data and factor data by DateKey
# =========================
df = pd.merge(y_data, x_data, on="DateKey", how="inner")

print("Merged shape:", df.shape)

# =========================
# 5. 删除关键变量缺失值
# =========================
required_cols = [
    "PERMNO", "MthCalDt", "DateKey",
    "MthRet", "MthPrc_Abs", "MthVol",
    "Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"
]

print("\nMissing values before drop:")
print(df[required_cols].isna().sum().sort_values(ascending=False))

df = df.dropna(subset=required_cols).copy()

# =========================
# Apply investability filters
# - Price must be at least 5
# - Trading volume must be greater than 0
# =========================
df = df[
    (df["MthPrc_Abs"] >= 5) &
    (df["MthVol"] > 0)
].copy()

# =========================
# Construct the liquidity measure DollarVol
# =========================
df["DollarVol"] = df["MthPrc_Abs"] * df["MthVol"]

print("\nAfter cleaning/filtering:", df.shape)
display(df.head())

Merged shape: (2960704, 18)

Missing values before drop:
MthRet        37443
MthPrc_Abs    36370
MthVol        36370
PERMNO            0
MthCalDt          0
DateKey           0
Mkt-RF            0
SMB               0
HML               0
RMW               0
CMA               0
RF                0
dtype: int64

After cleaning/filtering: (2368314, 19)


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthRet,MthVol,ShrOut,DateKey,MthPrc_Abs,Date,Mkt-RF,SMB,HML,RMW,CMA,RF,DollarVol
0,10001,36720410,EWST,7953,1995-01-31,7.750000,-0.031250,27232.0,2224,199501,7.750000,199501,0.0180,-0.0307,0.0259,0.0015,-0.0065,0.0042,211048.00000
1,10001,36720410,EWST,7953,1995-02-28,7.546875,-0.026210,21290.0,2224,199502,7.546875,199502,0.0363,-0.0049,0.0099,0.0061,-0.0040,0.0040,160672.96875
2,10001,36720410,EWST,7953,1995-03-31,7.500000,0.005971,8246.0,2244,199503,7.500000,199503,0.0219,-0.0053,-0.0213,-0.0013,0.0023,0.0046,61845.00000
3,10001,36720410,EWST,7953,1995-04-28,7.500000,0.000000,18812.0,2244,199504,7.500000,199504,0.0211,-0.0026,0.0167,0.0043,0.0082,0.0044,141090.00000
4,10001,36720410,EWST,7953,1995-05-31,7.875000,0.050000,19932.0,2244,199505,7.875000,199505,0.0290,-0.0217,0.0228,0.0041,0.0001,0.0054,156964.50000


In [31]:
# =========================
# Sort the data
# =========================
df = df.sort_values(["PERMNO", "MthCalDt"]).reset_index(drop=True)

# =========================
# 9.
# ExcessRet_t = MthRet_t - RF_t
# =========================
df["ExcessRet"] = df["MthRet"] - df["RF"]

# =========================
# Create the next-period target variable
# y_next = next month's excess return
# Apply shift(-1) within each PERMNO group
# =========================
df["y_next"] = df.groupby("PERMNO")["ExcessRet"].shift(-1)

# =========================
# Create the history length variable history_len
# This is used later to keep only stocks with at least 36 months of history
# =========================
df["history_len"] = df.groupby("PERMNO").cumcount() + 1

# =========================
# Keep only observations with at least 36 months of history
# Also drop the last observation for each stock if the next-period target is missing
# =========================
df_model = df[df["history_len"] >= 36].copy()
df_model = df_model.dropna(subset=["y_next"]).copy()

print("Final model data shape:", df_model.shape)

display(
    df_model[
        ["PERMNO", "MthCalDt", "DateKey", "MthRet", "RF", "ExcessRet", "y_next", "DollarVol", "history_len"]
    ].head(10)
)

Final model data shape: (1590627, 22)


,PERMNO,MthCalDt,DateKey,MthRet,RF,ExcessRet,y_next,DollarVol,history_len
35,10001,1997-12-31,199712,0.041502,0.0048,0.036702,-0.004300,1006893.000,36
36,10001,1998-01-30,199801,0.000000,0.0043,-0.004300,-0.010844,615924.000,37
37,10001,1998-02-27,199802,-0.006944,0.0039,-0.010844,-0.012702,126269.000,38
38,10001,1998-03-31,199803,-0.008802,0.0039,-0.012702,0.002843,676585.000,39
39,10001,1998-04-30,199804,0.007143,0.0043,0.002843,-0.018184,338629.125,40
40,10001,1998-05-29,199805,-0.014184,0.0040,-0.018184,0.001754,344285.625,41
41,10001,1998-06-30,199806,0.005854,0.0041,0.001754,0.010493,1009927.125,42
42,10001,1998-07-31,199807,0.014493,0.0040,0.010493,-0.004300,691048.750,43
43,10001,1998-08-31,199808,0.000000,0.0043,-0.004300,0.065686,401756.250,44
44,10001,1998-09-30,199809,0.070286,0.0046,0.065686,-0.003200,386825.750,45


In [32]:
# =========================
# Define the final feature columns and target variable
# =========================
feature_cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]
target_col = "y_next"

print("Features:", feature_cols)
print("Target:", target_col)

# Check whether there are still any missing values
print("\nMissing values in final dataset:")
print(df_model[feature_cols + [target_col]].isna().sum())

# Check how many stocks remain in each month
monthly_counts = (
    df_model.groupby("DateKey")["PERMNO"]
    .nunique()
    .reset_index(name="n_stocks")
)

print("\nMonthly stock count summary:")
print(monthly_counts["n_stocks"].describe())

display(monthly_counts.head())
display(monthly_counts.tail())


Features: ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
Target: y_next

Missing values in final dataset:
Mkt-RF    0
SMB       0
HML       0
RMW       0
CMA       0
y_next    0
dtype: int64

Monthly stock count summary:
count     352.000000
mean     4464.514205
std      1072.522085
min         1.000000
25%      4315.750000
50%      4514.000000
75%      4960.000000
max      5970.000000
Name: n_stocks, dtype: float64


,DateKey,n_stocks
0,199609,1
1,199610,1
2,199611,1
3,199612,2
4,199701,2


,DateKey,n_stocks
347,202508,5941
348,202509,5962
349,202510,5956
350,202511,5970
351,202512,599



Saved file: processed_panel_for_model.csv


In [33]:
# Save the cleaned final modeling dataset separately
output_path = "cleaned_model_data.csv"
df_model.to_csv(output_path, index=False)

print(f"Cleaned data saved to: {output_path}")
print("Final shape:", df_model.shape)

Cleaned data saved to: cleaned_model_data.csv
Final shape: (1590627, 22)
